# Unit 1 — Try working policies

Start with a working system. Run ACT, SmolVLA, and Diffusion Policy on the same task. Change a starting state or the SmolVLA prompt. Watch the video and record what you see.

This unit is a playground. It does not ask you to implement model code yet. Later units will replace one part of this system at a time.

**Time:** 15 minutes to inspect saved runs; 30+ minutes for live trials. The live cell is off by default. It loads local checkpoints and runs at most three development episodes. It does not download weights or data.

The local prepared weights are different recipes. The table below is saved evidence, not a new run. Compare actions and behavior as a first look. Unit 14 will use a controlled, paired evaluation.

In [ ]:
from __future__ import annotations


from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from collections.abc import Mapping
    from vla_lab.teaching.interfaces import ActionPolicy
    from vla_lab.teaching.interfaces import TensorBatch


import os
import sys
from pathlib import Path


def find_workspace(start: str | Path | None=None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    if current.is_file():
        current = current.parent
    for folder in (current, *current.parents):
        if (folder / "manifests" / "act_references.yaml").is_file():
            return folder
    raise FileNotFoundError("Open this notebook inside the VLA course workspace.")

WORKSPACE = find_workspace()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Use local caches and prevent model or data downloads when importing libraries.
HF_CACHE = WORKSPACE / ".local-data" / "hf"
(HF_CACHE / "hub").mkdir(parents=True, exist_ok=True)
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Workspace:", WORKSPACE)


Workspace: /home/michal/code/vlas_tutorial


## Saved evidence

Read the table as prior recorded results. ACT and SmolVLA used all 20 development states. The Diffusion row used states 0–4. These counts do not form a direct comparison because the evaluated state sets differ.

In [ ]:
import json

from IPython.display import Video, display

SAVED_RUNS = {
    "ACT": (".local-data/runs/act-bowl-plate-seeded-dev-all/summary.json", "19 / 20"),
    "SmolVLA": (".local-data/runs/smolvla-published-n10-seeded-dev-all/summary.json", "16 / 20"),
    "Diffusion Policy": (".local-data/runs/diffusion-2000-seeded-dev-20step-timed-0-4/summary.json", "2 / 5"),
}
for label, (relative_path, successes) in SAVED_RUNS.items():
    path = WORKSPACE / relative_path
    if path.is_file():
        record = json.loads(path.read_text())
        print(f"Saved evidence: {label}: {successes} successes; states {record['state_ids'][0]}–{record['state_ids'][-1]}; {path.relative_to(WORKSPACE)}")
    else:
        print(f"Saved evidence is not in this checkout: {label}: {relative_path}")

# This is one saved ACT video. It does not come from the live cell below.
saved_video = WORKSPACE / ".local-data/runs/act-bowl-plate-probe-200/rollout-state-000.mp4"
if saved_video.is_file():
    display(Video(str(saved_video), embed=True, mimetype="video/mp4", html_attributes="controls"))
else:
    print("The saved ACT video is not present in this checkout.")

Saved evidence: ACT: 19 / 20 successes; states 0–19; .local-data/runs/act-bowl-plate-seeded-dev-all/summary.json
Saved evidence: SmolVLA: 16 / 20 successes; states 0–19; .local-data/runs/smolvla-published-n10-seeded-dev-all/summary.json
Saved evidence: Diffusion Policy: 2 / 5 successes; states 0–4; .local-data/runs/diffusion-2000-seeded-dev-20step-timed-0-4/summary.json


## Choose a case

Try state 0, then another state from 0 to 9. These are development states for exploration. Change the prompt only to test SmolVLA. Keep a short note of the policy, state, prompt, outcome, and an observation about its motion.

In [7]:
TASK = "bowl_plate"
STATE_IDS = [0]                  # Try values from 0 to 9.
# METHODS = ["act", "smolvla_published", "diffusion"]
METHODS = ["smolvla_published"]
SMOLVLA_PROMPT = "put the grey bowl on the plate"
# SMOLVLA_PROMPT = "put a bowl on a plate"
RUN_LIVE = True                 # Set True to run the selected cases.
MAX_STEPS = 260

assert STATE_IDS and all(0 <= state <= 9 for state in STATE_IDS), "Use development states 0–9 here."
assert set(METHODS) <= {"act", "smolvla_published", "smolvla_learner", "diffusion"}
assert 1 <= MAX_STEPS <= 300

## Run prepared policies

Run one policy at a time. The loader checks each saved checkpoint against its manifest. The simulator records a video and result file under `.local-data/course-runs`. A repeated run uses a new output folder.

In [8]:
import gc
import json
from uuid import uuid4

import torch
from IPython.display import Video, display

from vla_lab.evaluation.demo import encode_rgb_video
from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.live import load_prepared_policy

live_results = []
for method in METHODS:
    try:
        policy, preprocessor, postprocessor, recipe = load_prepared_policy(method, WORKSPACE, task_alias=TASK)
    except Exception as error:
        # Keep one row per requested state even when loading fails.
        for state_id in STATE_IDS:
            output = WORKSPACE / ".local-data/course-runs/unit01" / f"{method}-{state_id}-{uuid4().hex[:8]}"
            output.mkdir(parents=True, exist_ok=False)
            row = {"method": method, "state": state_id, "success": False,
                    "steps": None, "termination": "policy_error",
                    "error": f"{type(error).__name__}: {error}", "video": None}
            (output / "result.json").write_text(json.dumps(row, indent=2) + "\n")
            live_results.append(row)
            print(row)
        continue
    for state_id in STATE_IDS:
        output = WORKSPACE / ".local-data/course-runs/unit01" / f"{method}-{state_id}-{uuid4().hex[:8]}"
        output.mkdir(parents=True, exist_ok=False)
        frames = []
        class PredictionLog:
            def __init__(self, wrapped: ActionPolicy) -> None: self.wrapped = wrapped; self.actions = []
            def reset(self) -> None: self.wrapped.reset()
            def select_action(self, batch: TensorBatch) -> torch.Tensor:
                action = self.wrapped.select_action(batch)
                self.actions.append(action.detach().cpu().reshape(-1).tolist())
                return action
        logged_policy = PredictionLog(policy)
        env = None
        try:
            env = open_libero_task(TASK, state_id, WORKSPACE / "configs", WORKSPACE / ".local-data")
            result = run_policy_episode(
                env, logged_policy, preprocessor, postprocessor, TASK, state_id,
                WORKSPACE / "configs", partition="development", max_steps=MAX_STEPS,
                on_frame=frames.append,
                task_prompt=SMOLVLA_PROMPT if method in ("smolvla_published", "smolvla_learner") else None,
                reset_seed=20260930 + state_id,
            )
            video = encode_rgb_video(frames, output / "rollout.mp4", fps=20) if frames else None
            row = {"method": method, "state": state_id, "success": result.success,
                    "steps": result.steps, "termination": result.termination_reason,
                    "error": None, "video": str(video) if video else None,
                    "normalized_predicted_actions": logged_policy.actions,
                    "prompt": SMOLVLA_PROMPT if method in ("smolvla_published", "smolvla_learner") else None}
        except Exception as error:
            row = {"method": method, "state": state_id, "success": False,
                    "steps": None, "termination": "infrastructure_error",
                    "error": f"{type(error).__name__}: {error}", "video": None,
                    "normalized_predicted_actions": logged_policy.actions}
        finally:
            if env is not None:
                env.close()
        (output / "result.json").write_text(json.dumps(row, indent=2) + "\n")
        live_results.append(row)
        print({key: value for key, value in row.items() if key != "normalized_predicted_actions"})
        if row.get("video"):
            display(Video(row["video"], embed=True, mimetype="video/mp4", html_attributes="controls"))
        del logged_policy, frames, env
    del policy, preprocessor, postprocessor, recipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'method': 'smolvla_published', 'state': 0, 'success': True, 'steps': 78, 'termination': 'success', 'error': None, 'video': '/home/michal/code/vlas_tutorial/.local-data/course-runs/unit01/smolvla_published-0-c1ba58dc/rollout.mp4', 'prompt': 'put the grey bowl on the plate'}


## Record one observation

For each run, write one sentence about the approach, one sentence about what failed or worked, and one question to investigate later. Keep these notes separate from the saved evidence above.

In [ ]:
observations = []  # Add rows such as {"method": "act", "state": 0, "note": "..."}.
observations